# Sprint 9 - Dual-Head with Separate Backbones

**Why this sprint:** Sprint 8 proved the bottleneck is the *shared backbone*, not the head.
With one frozen backbone, the field head capped at 0.41 (Sprint 8) even with a domain
classifier. The fix: give each head its OWN backbone. `backbone_lab` is trained on
PlantVillage; `backbone_field` is warm-started from it then adapted on PlantDoc. No shared
backbone = no interference.

**Architecture:**
```
backbone_lab   (PV-trained)    -> head_lab
backbone_field (PD-adapted)    -> head_field
domain_classifier (on backbone_lab features) -> lab=0 / field=1
```

**Sprint 9 gates (must BOTH pass):**
1. **Field:** head_field on PlantDoc F1 >= 0.60
2. **Lab:** head_lab on PlantVillage F1 >= 0.85

In [ ]:
import os
import platform
import subprocess
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
    print(gpu.stdout.strip().splitlines()[0] if gpu.stdout.strip() else gpu.stderr.strip() or "No GPU detected (CPU only)")
except Exception as exc:
    print("GPU check skipped:", exc)

## Step 1 - Mount Drive + clone repo

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

REPO_URL = "https://github.com/io-PEAK/folium.git"
REPO_DIR = Path("/content/folium")
DATA_DIR = Path("/content/drive/MyDrive/folium/data")
LOCAL_RAW_DIR = Path("/content/folium_raw")
LOCAL_DATA_DIR = Path("/content/folium_data")
CHECKPOINT_DIR = Path("/content/drive/MyDrive/folium/checkpoints")
RESULTS_DIR = Path("/content/drive/MyDrive/folium/results")

if not (REPO_DIR / "ml").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only -q

for d in (DATA_DIR, LOCAL_RAW_DIR, LOCAL_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("DATA_DIR (archives):", DATA_DIR)
print("LOCAL_DATA_DIR:", LOCAL_DATA_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

def run(cmd, cwd, label, stream=False):
    """Run a subprocess. If stream=True, print stdout line-by-line in real time."""
    env = dict(os.environ, PYTHONPATH=str(REPO_DIR))
    if stream:
        proc = subprocess.Popen(
            cmd, cwd=str(cwd), env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        lines = []
        for line in proc.stdout:
            print(line, end="")
            lines.append(line)
        proc.wait()
        combined = "".join(lines)
        if proc.returncode != 0:
            print(f"\n[{label}] failed (returncode {proc.returncode})")
        assert proc.returncode == 0, label
        class _Result:
            pass
        r = _Result()
        r.stdout = combined
        r.stderr = ""
        r.returncode = 0
        return r
    result = subprocess.run(cmd, cwd=str(cwd), env=env, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[{label}] failed (returncode {result.returncode})")
        print("stdout tail:\n", result.stdout[-2000:])
        print("stderr tail:\n", result.stderr[-2000:])
    assert result.returncode == 0, label
    return result

## Step 2 - Install dependencies

In [ ]:
%pip install -q --upgrade pip
%pip install -q torch torchvision albumentations matplotlib pandas tqdm opencv-python-headless scikit-learn

## Step 2b - Clean old Sprint 8/9 rows from ablation CSV

In [ ]:
import pandas as pd

csv_path = RESULTS_DIR / "ablation_results.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    old = df["variant"].str.startswith("s8_") | df["variant"].str.startswith("s9_")
    n_old = old.sum()
    if n_old > 0:
        df = df[~old].reset_index(drop=True)
        df.to_csv(csv_path, index=False)
        print(f"Removed {n_old} old s8/s9 rows from {csv_path}")
    else:
        print("No old s8/s9 rows found.")
else:
    print("No ablation CSV yet.")

## Step 3 - Hydrate raw from Drive, then organize splits locally

In [ ]:
import sys

sys.path.insert(0, str(REPO_DIR))
from scripts.download_datasets import PLANTVILLAGE_EXPECTED, PLANTDOC_EXPECTED, hydrate_dataset

for name, expected in (("plantvillage", PLANTVILLAGE_EXPECTED), ("plantdoc", PLANTDOC_EXPECTED)):
    try:
        hydrate_dataset(LOCAL_RAW_DIR, DATA_DIR, name, expected)
    except RuntimeError as exc:
        print("HYDRATE FAILED:", exc)
        raise

result = run([
    sys.executable,
    str(REPO_DIR / "scripts" / "organize_datasets.py"),
    "--raw-dir", str(LOCAL_RAW_DIR),
    "--data-dir", str(LOCAL_DATA_DIR),
    "--seed", "42",
    "--val-fraction", "0.15",
    "--test-fraction", "0.15",
], cwd=str(REPO_DIR), label="organize_datasets.py failed")
print("Data ready at", LOCAL_DATA_DIR)

## Step 4 - Train backbone_lab + head_lab on PlantVillage

Train backbone_lab and head_lab on lab photos. Everything else frozen.

In [ ]:
cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantvillage",
    "--backbone", "resnet50",
    "--epochs", "5",
    "--lr", "1e-3",
    "--augment",
    "--dual-head",
    "--separate-backbones",
    "--train-head", "lab",
    "--tag", "s9_lab",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
]
result = run(cmd, cwd=str(REPO_DIR), label="s9 lab training failed", stream=True)
print("\n=== Lab backbone+head training complete ===")

## Step 5 - Train backbone_field + head_field on PlantDoc

Warm-start from the lab checkpoint: copy backbone_lab -> backbone_field and
head_lab -> head_field. Then train backbone_field + head_field on PlantDoc.
This adapts the field backbone to real-world photos.

In [ ]:
S9_LAB_CKPT = CHECKPOINT_DIR / "best_plantvillage_s9_lab.pt"
assert S9_LAB_CKPT.exists(), f"Missing lab checkpoint: {S9_LAB_CKPT}. Run Step 4 first."

cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantdoc",
    "--map-to-pv",
    "--backbone", "resnet50",
    "--epochs", "10",
    "--lr", "1e-3",
    "--augment",
    "--dual-head",
    "--separate-backbones",
    "--train-head", "field",
    "--init-from", str(S9_LAB_CKPT),
    "--tag", "s9_field",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
]
result = run(cmd, cwd=str(REPO_DIR), label="s9 field training failed", stream=True)
print("\n=== Field backbone+head training complete ===")

## Step 6 - Train domain classifier

A single linear layer (2048 -> 2) that distinguishes lab from field photos.
Trained on backbone_lab features of PlantVillage (label=0) + PlantDoc (label=1).

In [ ]:
S9_FIELD_CKPT = CHECKPOINT_DIR / "best_plantdoc_s9_field.pt"
assert S9_FIELD_CKPT.exists(), f"Missing field checkpoint: {S9_FIELD_CKPT}. Run Step 5 first."

cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--backbone", "resnet50",
    "--epochs", "3",
    "--lr", "1e-3",
    "--augment",
    "--dual-head",
    "--separate-backbones",
    "--train-head", "domain",
    "--init-from", str(S9_FIELD_CKPT),
    "--tag", "s9_domain",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
]
result = run(cmd, cwd=str(REPO_DIR), label="s9 domain training failed", stream=True)
print("\n=== Domain classifier training complete ===")

## Step 7 - t-SNE of backbone features (domain shift figure)

Extract backbone_lab features from a sample of PlantVillage (lab) and PlantDoc
(field) test images, then t-SNE. The two domains should form separate clusters
this is the domain shift that makes a single shared model fail.

In [ ]:
import numpy as np
import torch
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from ml.data_loading import build_loaders
from ml.model import build_dual_head_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DOMAIN_CKPT = CHECKPOINT_DIR / "best_domain_s9_domain.pt"
ckpt = torch.load(DOMAIN_CKPT, map_location="cpu")
model = build_dual_head_model(
    num_classes=ckpt["model_kwargs"]["num_classes"],
    backbone=ckpt["model_kwargs"]["backbone"],
    separate_backbones=ckpt["model_kwargs"].get("separate_backbones", False),
)
model.load_state_dict(ckpt["state_dict"], strict=False)
model.to(device).eval()

def feats_for(dataset, n=150):
    _, _, test_loader, _ = build_loaders(
        LOCAL_DATA_DIR, dataset=dataset, batch_size=32, num_workers=2, seed=42,
        map_to_pv=(dataset == "plantdoc"),
    )
    out = []
    with torch.no_grad():
        for images, _ in test_loader:
            f = model.extract_features(images.to(device)).cpu().numpy()
            out.append(f)
            if len(out) * 32 >= n:
                break
    return np.concatenate(out)[:n]

pv = feats_for("plantvillage")
pd = feats_for("plantdoc")
all_feats = np.concatenate([pv, pd])
labels = np.array([0] * len(pv) + [1] * len(pd))

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
emb = tsne.fit_transform(all_feats)

plt.figure(figsize=(7, 6))
plt.scatter(emb[:len(pv), 0], emb[:len(pv), 1], s=10, alpha=0.5, label="lab (PlantVillage)")
plt.scatter(emb[len(pv):, 0], emb[len(pv):, 1], s=10, alpha=0.5, label="field (PlantDoc)")
plt.legend()
plt.title("Backbone feature space: lab vs field (domain shift)")
plt.xlabel("t-SNE 1"); plt.ylabel("t-SNE 2")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "tsne_domain_shift.png", dpi=120)
print("Saved", RESULTS_DIR / "tsne_domain_shift.png")
print(f"PV features: {pv.shape}, PD features: {pd.shape}")

## Step 8 - Label audit (PlantDoc noise figure)

Score every PlantDoc image with the field head, sorted least-confident first.
Suspected mislabels surface at the top. This quantifies the dataset ceiling.

In [ ]:
AUDIT_OUT = RESULTS_DIR / "plantdoc_label_audit_s9.csv"
cmd = [
    sys.executable, str(REPO_DIR / "scripts" / "audit_plantdoc_labels.py"),
    "--checkpoint", str(DOMAIN_CKPT),
    "--data-dir", str(LOCAL_DATA_DIR),
    "--split", "test",
    "--dual-head",
    "--separate-backbones",
    "--out", str(AUDIT_OUT),
]
result = run(cmd, cwd=str(REPO_DIR), label="label audit failed", stream=True)

## Step 9 - Evaluate

Run 6 evaluations:
1. head_lab alone on PlantVillage (true lab score)
2. head_field alone on PlantDoc (true field score)
3. domain-routed on PlantVillage (production inference)
4. domain-routed on PlantDoc (production inference)
5. predict_dual on both (old confidence-race, for comparison)

In [ ]:
DOMAIN_CKPT = CHECKPOINT_DIR / "best_domain_s9_domain.pt"
assert DOMAIN_CKPT.exists(), f"Missing domain checkpoint: {DOMAIN_CKPT}. Run Step 6 first."

def eval_cmd(dataset, extra):
    return [
        sys.executable, "-m", "ml.evaluate",
        "--checkpoint", str(DOMAIN_CKPT),
        "--data-dir", str(LOCAL_DATA_DIR),
        "--dataset", dataset,
        "--split", "test",
        "--dual-head",
        "--separate-backbones",
        *extra,
        "--results", str(RESULTS_DIR / "ablation_results.csv"),
    ]

# 1) head_lab on PlantVillage
print("=" * 60)
print("9a) head_lab on PlantVillage (true lab score)")
print("=" * 60)
result = run(eval_cmd("plantvillage", ["--eval-head", "lab", "--variant", "s9_lab_on_plantvillage"]), cwd=str(REPO_DIR), label="eval failed", stream=True)

# 2) head_field on PlantDoc
print("\n" + "=" * 60)
print("9b) head_field on PlantDoc (true field score)")
print("=" * 60)
result = run(eval_cmd("plantdoc", ["--map-to-pv", "--eval-head", "field", "--variant", "s9_field_on_plantdoc"]), cwd=str(REPO_DIR), label="eval failed", stream=True)

# 3) domain-routed on PlantVillage
print("\n" + "=" * 60)
print("9c) domain-routed on PlantVillage")
print("=" * 60)
result = run(eval_cmd("plantvillage", ["--predict-mode", "routed", "--variant", "s9_routed_plantvillage"]), cwd=str(REPO_DIR), label="eval failed", stream=True)

# 4) domain-routed on PlantDoc
print("\n" + "=" * 60)
print("9d) domain-routed on PlantDoc")
print("=" * 60)
result = run(eval_cmd("plantdoc", ["--map-to-pv", "--predict-mode", "routed", "--variant", "s9_routed_plantdoc"]), cwd=str(REPO_DIR), label="eval failed", stream=True)

# 5) predict_dual on PlantVillage
print("\n" + "=" * 60)
print("9e) predict_dual on PlantVillage (confidence-race, comparison)")
print("=" * 60)
result = run(eval_cmd("plantvillage", ["--predict-mode", "dual", "--variant", "s9_dual_plantvillage"]), cwd=str(REPO_DIR), label="eval failed", stream=True)

# 6) predict_dual on PlantDoc
print("\n" + "=" * 60)
print("9f) predict_dual on PlantDoc (confidence-race, comparison)")
print("=" * 60)
result = run(eval_cmd("plantdoc", ["--map-to-pv", "--predict-mode", "dual", "--variant", "s9_dual_plantdoc"]), cwd=str(REPO_DIR), label="eval failed", stream=True)

## Step 10 - The verdict

Per-head scores are ground truth. Routed scores show production performance.

**Gates (per-head scores):**
- head_field on PlantDoc F1 >= 0.60
- head_lab on PlantVillage F1 >= 0.85

In [ ]:
import pandas as pd

df = pd.read_csv(str(RESULTS_DIR / "ablation_results.csv")).drop_duplicates()

print("=== Per-head scores (ground truth) ===")
per_head_keys = ["s9_lab_on_plantvillage", "s9_field_on_plantdoc"]
per_head = df[df["variant"].isin(per_head_keys)].copy()
if len(per_head) > 0:
    print(per_head[["variant", "f1"]].to_string(index=False))

print("\n=== Domain-routed scores (production) ===")
routed_keys = ["s9_routed_plantvillage", "s9_routed_plantdoc"]
routed = df[df["variant"].isin(routed_keys)].copy()
if len(routed) > 0:
    print(routed[["variant", "f1"]].to_string(index=False))

if len(per_head) == 2:
    field_f1 = per_head[per_head["variant"] == "s9_field_on_plantdoc"]["f1"].iloc[0]
    lab_f1 = per_head[per_head["variant"] == "s9_lab_on_plantvillage"]["f1"].iloc[0]
    field_pass = "PASS" if field_f1 >= 0.60 else "FAIL"
    lab_pass = "PASS" if lab_f1 >= 0.85 else "FAIL"
    print(f"\nVerdict: field {field_f1:.4f} -> {field_pass} | lab {lab_f1:.4f} -> {lab_pass}")
else:
    print("\nPer-head rows not found. Run Step 9 first.")